# ETL Pipeline — feeds → extraction → macro indicators → market data

Runs the same four modules that deploy as separate Cloud Functions (`etl/ingest_feeds.py`, `etl/extract_and_tag.py`, `etl/ingest_macro_indicators.py`, `etl/ingest_market_data.py`), in the order the pipeline actually depends on them, in one notebook. Nothing is reimplemented here — every cell below calls the real deployable function; this notebook is the interactive/documentation layer, the `.py` modules stay the source of truth Cloud Functions deploy from.

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root
from dotenv import load_dotenv
load_dotenv(pathlib.Path.cwd().parent / '.env')

import pandas as pd

## 1. RSS ingestion — `etl/ingest_feeds.py`

Pulls every `status: active` feed in `feeds.yaml`, dedupes against previously seen article URLs, writes raw article JSON. Set `USE_LOCAL_FEEDS=true` (and run `python -m etl.mock_rss_server` in another terminal first) to reroute this at synthetic feeds for volume testing instead of real ones — the cell below uses real feeds.

In [ ]:
from etl.ingest_feeds import load_feeds, run_ingest
pd.DataFrame(load_feeds())[['category', 'name', 'url']]

In [ ]:
feeds_result = run_ingest()
feeds_result

## 2. Structured extraction — `etl/extract_and_tag.py`

Calls the Gemini API once per raw article for structured extraction. **Requires `GEMINI_API_KEY`** in `.env`. A new key's project free tier is a provisional 20 requests/day — easy to exhaust while testing; failures are dead-lettered (see cell below) rather than silently lost.

In [ ]:
assert os.environ.get('GEMINI_API_KEY'), 'Set GEMINI_API_KEY in .env before running this cell'
from etl.extract_and_tag import run_extraction, Extraction, SECTOR_IDS, POLICY_SUBTYPES, MECHANISM_IDS
Extraction.model_json_schema()

In [ ]:
extract_result = run_extraction()
extract_result

In [ ]:
deadletter = pathlib.Path.cwd().parent / 'data' / 'cache' / 'extract_deadletter.jsonl'
if deadletter.exists():
    lines = deadletter.read_text().splitlines()
    print(f'{len(lines)} dead-lettered articles; most recent:')
    print(lines[-1] if lines else '(empty)')
else:
    print('no dead-letter file yet')

## 3. Macro indicators — `etl/ingest_macro_indicators.py`

BIS (policy rate, REER, no key needed) + World Bank WDI (CPI, GDP growth, unemployment, trade balance, gov debt, FDI, no key needed) + FRED (global benchmarks, needs `FRED_API_KEY`), then derives `real_interest_rate = policy_rate - cpi_inflation`. Coverage is uneven by design — BIS doesn't cover Taiwan/Zambia/DR Congo — skipped with a logged reason, not backfilled.

In [ ]:
from etl.ingest_macro_indicators import run_ingest as run_macro_ingest, WDI_INDICATORS, FRED_SERIES
pd.DataFrame(list(WDI_INDICATORS.items()), columns=['indicator_id', 'World Bank WDI code'])

In [ ]:
macro_result = run_macro_ingest()
macro_result

## 4. Commodity price & equity valuation — `etl/ingest_market_data.py`

World Bank Pink Sheet (copper, nickel only — no equivalent free spot price found for lithium/cobalt/rare earths) + Yahoo Finance sector ETF proxies (all six sectors).

In [ ]:
from etl.ingest_market_data import ingest_commodity_prices, ingest_equity_valuations, SECTOR_ETF, PINK_SHEET_COLUMNS
pd.DataFrame(list(SECTOR_ETF.items()), columns=['sector_id', 'ETF ticker'])

In [ ]:
price_rows = ingest_commodity_prices()
valuation_rows = ingest_equity_valuations()
price_rows, valuation_rows

## Next step

Once all four steps above have run at least once, `02_correlation_regression.ipynb` joins this data and runs the Pingouin analysis.